In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Load the Loan Approval dataset

import os

candidate_paths = [
    "/kaggle/input/datasets/anshgoel0110/loan-approval-project/Loan.csv",
    "/kaggle/input/loan-approval-project/Loan.csv",
    "Loan.csv",
    "/mnt/data/Loan.csv"
]

data_path = next(
    (path for path in candidate_paths if os.path.exists(path)),
    None
)

if data_path is None:
    # Search Kaggle input folders as a fallback.
    kaggle_matches = []
    for root, _, files in os.walk("/kaggle/input"):
        if "Loan.csv" in files:
            kaggle_matches.append(
                os.path.join(root, "Loan.csv")
            )

    if kaggle_matches:
        data_path = kaggle_matches[0]

if data_path is None:
    raise FileNotFoundError(
        "Loan.csv was not found. Attach the Loan Approval dataset "
        "to Kaggle or place Loan.csv in the notebook working directory."
    )

df = pd.read_csv(data_path)

print("Dataset path:", data_path)
print("Dataset shape:", df.shape)


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

# Handle Missing Values

# Missing Value Analysis

**Important correction:** Missing-value statistics are inspected before modeling, but no imputation is fitted on the complete dataset.  
All imputation used for machine learning is fitted **only on the training set** to avoid data leakage.


In [ ]:
# Check missing values without modifying the dataset
missing_summary = df.isnull().sum().sort_values(ascending=False)
missing_summary[missing_summary > 0]


In [ ]:
# Separate columns by type for EDA only
numerical_columns = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns:", len(numerical_columns))
print("Categorical columns:", len(categorical_columns))


# EDA - Exploratory Data Analysis

In [ ]:
classes_count = df["LoanApproved"].value_counts()

plt.pie(classes_count, labels=["No","Yes"], autopct="%1.1f%%")
plt.title("is loan approved or not")
plt.savefig("loan_approval_distribution.png", dpi=300, bbox_inches="tight")

In [ ]:
gend_count = df["MaritalStatus"].value_counts()
ax = sns.barplot(gend_count)
ax.bar_label(ax.containers[0])

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df["AnnualIncome"], bins=30)

plt.title("Annual Income Distribution")

plt.xlabel("Annual Income")

plt.ylabel("Frequency")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df["CreditScore"], bins=30)

plt.title("Credit Score Distribution")

plt.xlabel("Credit Score")

plt.ylabel("Frequency")

plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

sns.boxplot(
    data=df,
    x="LoanApproved",
    y="AnnualIncome"
)

plt.title("Annual Income vs Loan Approval")
plt.xlabel("Loan Approved")
plt.ylabel("Annual Income")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

sns.boxplot(
    data=df,
    x="LoanApproved",
    y="DebtToIncomeRatio"
)

plt.title("Debt-to-Income Ratio vs Loan Approval")
plt.xlabel("Loan Approved")
plt.ylabel("Debt-to-Income Ratio")
plt.savefig("dti_vs_approval.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

sns.boxplot(
    data=df,
    x="LoanApproved",
    y="CreditScore"
)

plt.title("Credit Score vs Loan Approval")
plt.xlabel("Loan Approved")
plt.ylabel("Credit Score")
plt.savefig("credit_score_vs_approval.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

sns.boxplot(
    data=df,
    x="LoanApproved",
    y="RiskScore"
)

plt.title("Risk Score vs Loan Approval - Leakage Audit")
plt.xlabel("Loan Approved")
plt.ylabel("Risk Score")
plt.tight_layout()
plt.savefig(
    "risk_score_leakage_audit.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
sns.histplot(
    data = df,
    x = "CreditScore",
    hue = "LoanApproved",
    bins = 20,
    multiple = "dodge"
)

# Categorical Feature Definition

Categorical variables are defined here but **not fitted/encoded yet**.  
The encoder will be fitted only on the training split later.


In [ ]:
categorical_cols = [
    "EmploymentStatus",
    "EducationLevel",
    "LoanPurpose",
    "HomeOwnershipStatus",
    "MaritalStatus"
]

print("Categorical columns:", categorical_cols)


In [ ]:
# Remove date field from the modeling dataset.
# ApplicationDate is not used as a predictive feature.
df = df.drop(columns=["ApplicationDate"]).copy()

print("Dataset shape after removing ApplicationDate:", df.shape)


# Correlation Heatmap

This visualization is used only for exploratory analysis.  
The **RiskScore** variable is inspected because it has a strong relationship with the target, but it is **excluded from the predictive model** to avoid possible target leakage and to keep risk assessment as an output of the system.


In [ ]:
num_cols = df.select_dtypes(include="number")
corr_matrix = num_cols.corr()

plt.figure(figsize=(24, 12))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
target_correlation = (
    num_cols.corr()["LoanApproved"]
    .sort_values(ascending=False)
)

target_correlation


# Leakage Audit

`RiskScore` is deliberately **not** used as a model input in the corrected experiment.

Reason:
- it is a risk-related variable with a strong association with `LoanApproved`;
- using a variable that may encode the outcome can make test performance unrealistically high;
- risk should instead be derived from the model's predicted rejection probability after prediction.

This notebook therefore follows the safer direction:

**Applicant / financial / credit information → ML prediction → risk assessment.**


In [ ]:
TARGET = "LoanApproved"
LEAKAGE_FEATURES = ["RiskScore"]

risk_score_correlation = df["RiskScore"].corr(df[TARGET])

print(f"RiskScore correlation with {TARGET}: {risk_score_correlation:.4f}")
print("RiskScore will NOT be used as a predictive feature.")


# Train-Test Split and Leakage-Safe Preprocessing

The corrected pipeline performs the following steps:

1. Remove the target and RiskScore.
2. Create non-target-based engineered features.
3. Split the data using stratification.
4. Fit imputers only on training data.
5. Fit OneHotEncoder only on training data.
6. Fit StandardScaler only on training data.
7. Transform the test data using the already-fitted training objects.

This prevents information from the test set from influencing preprocessing.


In [ ]:
# -----------------------------
# Feature matrix and target
# -----------------------------

X_raw = df.drop(
    columns=[TARGET] + LEAKAGE_FEATURES
).copy()

y = df[TARGET].astype(int).copy()

# -----------------------------
# Feature engineering
# -----------------------------
# These transformations use only the applicant's own feature values
# and do not use the target.

X_raw["DebtToIncomeRatio_sq"] = X_raw["DebtToIncomeRatio"] ** 2
X_raw["CreditScore_sq"] = X_raw["CreditScore"] ** 2

# Remove the original variables after creating their engineered versions.
X_raw = X_raw.drop(
    columns=["CreditScore", "DebtToIncomeRatio"]
)

print("Final raw feature count before encoding:", X_raw.shape[1])
print("RiskScore present:", "RiskScore" in X_raw.columns)


In [ ]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train_raw.shape[0])
print("Testing rows:", X_test_raw.shape[0])
print("Training class distribution:")
print(y_train.value_counts())
print("Testing class distribution:")
print(y_test.value_counts())


In [ ]:
# Use an explicit, fixed categorical order so the saved OHE
# categories match the Streamlit application exactly.

model_categorical_cols = [
    "EmploymentStatus",
    "EducationLevel",
    "LoanPurpose",
    "HomeOwnershipStatus",
    "MaritalStatus"
]

model_numerical_cols = [
    column
    for column in X_train_raw.columns
    if column not in model_categorical_cols
]

print("Numerical features:", len(model_numerical_cols))
print("Categorical features:", model_categorical_cols)


In [ ]:
# -----------------------------
# Fit imputers ONLY on training data
# -----------------------------

num_imp = SimpleImputer(strategy="mean")
cat_imp = SimpleImputer(strategy="most_frequent")

X_train_num = pd.DataFrame(
    num_imp.fit_transform(X_train_raw[model_numerical_cols]),
    columns=model_numerical_cols,
    index=X_train_raw.index
)

X_test_num = pd.DataFrame(
    num_imp.transform(X_test_raw[model_numerical_cols]),
    columns=model_numerical_cols,
    index=X_test_raw.index
)

X_train_cat = pd.DataFrame(
    cat_imp.fit_transform(X_train_raw[model_categorical_cols]),
    columns=model_categorical_cols,
    index=X_train_raw.index
)

X_test_cat = pd.DataFrame(
    cat_imp.transform(X_test_raw[model_categorical_cols]),
    columns=model_categorical_cols,
    index=X_test_raw.index
)

print("Training-only imputers fitted successfully.")


In [ ]:
# -----------------------------
# Fit OneHotEncoder ONLY on training data
# -----------------------------

ohe = OneHotEncoder(
    drop="first",
    sparse_output=False,
    handle_unknown="ignore"
)

X_train_encoded = ohe.fit_transform(X_train_cat)
X_test_encoded = ohe.transform(X_test_cat)

encoded_columns = ohe.get_feature_names_out(
    model_categorical_cols
)

X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=encoded_columns,
    index=X_train_raw.index
)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=encoded_columns,
    index=X_test_raw.index
)

print("One-hot encoder fitted using training data only.")


In [ ]:
# Combine numerical and encoded categorical features
X_train_processed = pd.concat(
    [X_train_num, X_train_encoded],
    axis=1
)

X_test_processed = pd.concat(
    [X_test_num, X_test_encoded],
    axis=1
)

# Keep exactly the same column order in train and test
X_test_processed = X_test_processed.reindex(
    columns=X_train_processed.columns,
    fill_value=0
)

feature_columns = X_train_processed.columns.tolist()

print("Processed feature count:", len(feature_columns))
print("RiskScore present:", any("RiskScore" in c for c in feature_columns))


In [ ]:
# -----------------------------
# Fit scaler ONLY on training data
# -----------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_processed)
X_test_scaled = scaler.transform(X_test_processed)

print("Scaling completed without test-set fitting.")
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)


# Train and Evaluate Nine Classification Models

The same leakage-safe training/test representation is used for all nine classifiers:

- Logistic Regression
- K-Nearest Neighbors
- Gaussian Naive Bayes
- Decision Tree
- Random Forest
- Support Vector Machine
- Gradient Boosting
- XGBoost
- CatBoost

Evaluation includes Accuracy, Precision, Recall, F1-score and ROC-AUC.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

def evaluate_model(model_name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    y_pred = np.asarray(model.predict(X_test)).ravel()

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = None

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan
    }

    return metrics, y_pred, y_prob, confusion_matrix(y_test, y_pred)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    "K-Nearest Neighbors": KNeighborsClassifier(
        n_neighbors=5
    ),
    "Gaussian Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(
        random_state=42,
        max_depth=8
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "Support Vector Machine": SVC(
        probability=True,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=4,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=0
    )
}

all_model_metrics = []
model_predictions = {}
model_probabilities = {}
model_confusion_matrices = {}

for name, model in models.items():
    metrics, y_pred, y_prob, cm = evaluate_model(
        name,
        model,
        X_train_scaled,
        X_test_scaled,
        y_train,
        y_test
    )

    all_model_metrics.append(metrics)
    model_predictions[name] = y_pred
    model_probabilities[name] = y_prob
    model_confusion_matrices[name] = cm

results_df = (
    pd.DataFrame(all_model_metrics)
    .sort_values(
        by=["F1 Score", "ROC-AUC"],
        ascending=False
    )
    .reset_index(drop=True)
)

results_df


In [ ]:
# Visual comparison of model performance
plot_metrics = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]

ax = results_df.set_index("Model")[plot_metrics].plot(
    kind="bar",
    figsize=(14, 6)
)

ax.set_title("Performance Comparison of Machine Learning Models")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(
    "model_performance_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
# CatBoost confusion matrix
catboost_cm = model_confusion_matrices["CatBoost"]

plt.figure(figsize=(6, 5))
sns.heatmap(
    catboost_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Rejected", "Approved"],
    yticklabels=["Rejected", "Approved"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("CatBoost Confusion Matrix")
plt.tight_layout()
plt.savefig(
    "catboost_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


# Final CatBoost Model

CatBoost is retained as the final model for the CreditWise framework so that the deployment and XAI stages use one consistent model.

The model is evaluated using the held-out test set and then validated using **leakage-safe 5-fold stratified cross-validation**. The cross-validation preprocessing is fitted separately inside each fold.


In [ ]:
# Re-create the final CatBoost model independently
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

catboost_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=0
)

catboost_model.fit(
    X_train_scaled,
    y_train
)

catboost_test_pred = np.asarray(
    catboost_model.predict(X_test_scaled)
).ravel()

catboost_test_prob = catboost_model.predict_proba(
    X_test_scaled
)[:, 1]

catboost_test_metrics = {
    "Accuracy": accuracy_score(y_test, catboost_test_pred),
    "Precision": precision_score(y_test, catboost_test_pred, zero_division=0),
    "Recall": recall_score(y_test, catboost_test_pred, zero_division=0),
    "F1 Score": f1_score(y_test, catboost_test_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, catboost_test_prob)
}

print("CatBoost held-out test performance:")
for metric, value in catboost_test_metrics.items():
    print(f"{metric}: {value:.6f}")


In [ ]:
# Leakage-safe 5-fold cross-validation without relying on
# sklearn Pipeline/CatBoost sklearn-tag compatibility.

from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_records = []

for fold_number, (train_idx, valid_idx) in enumerate(
    cv.split(X_train_raw, y_train),
    start=1
):
    X_fold_train = X_train_raw.iloc[train_idx].copy()
    X_fold_valid = X_train_raw.iloc[valid_idx].copy()

    y_fold_train = y_train.iloc[train_idx]
    y_fold_valid = y_train.iloc[valid_idx]

    # Fit numerical imputer on this fold's training data only
    fold_num_imp = SimpleImputer(strategy="mean")
    fold_cat_imp = SimpleImputer(strategy="most_frequent")

    fold_train_num = pd.DataFrame(
        fold_num_imp.fit_transform(
            X_fold_train[model_numerical_cols]
        ),
        columns=model_numerical_cols,
        index=X_fold_train.index
    )

    fold_valid_num = pd.DataFrame(
        fold_num_imp.transform(
            X_fold_valid[model_numerical_cols]
        ),
        columns=model_numerical_cols,
        index=X_fold_valid.index
    )

    # Fit categorical imputer on this fold's training data only
    fold_train_cat = pd.DataFrame(
        fold_cat_imp.fit_transform(
            X_fold_train[model_categorical_cols]
        ),
        columns=model_categorical_cols,
        index=X_fold_train.index
    )

    fold_valid_cat = pd.DataFrame(
        fold_cat_imp.transform(
            X_fold_valid[model_categorical_cols]
        ),
        columns=model_categorical_cols,
        index=X_fold_valid.index
    )

    # Fit OHE on this fold's training data only
    fold_ohe = OneHotEncoder(
        drop="first",
        sparse_output=False,
        handle_unknown="ignore"
    )

    fold_train_encoded = fold_ohe.fit_transform(
        fold_train_cat
    )

    fold_valid_encoded = fold_ohe.transform(
        fold_valid_cat
    )

    fold_encoded_columns = fold_ohe.get_feature_names_out(
        model_categorical_cols
    )

    fold_train_encoded = pd.DataFrame(
        fold_train_encoded,
        columns=fold_encoded_columns,
        index=X_fold_train.index
    )

    fold_valid_encoded = pd.DataFrame(
        fold_valid_encoded,
        columns=fold_encoded_columns,
        index=X_fold_valid.index
    )

    # Combine features
    fold_train_processed = pd.concat(
        [fold_train_num, fold_train_encoded],
        axis=1
    )

    fold_valid_processed = pd.concat(
        [fold_valid_num, fold_valid_encoded],
        axis=1
    )

    fold_valid_processed = fold_valid_processed.reindex(
        columns=fold_train_processed.columns,
        fill_value=0
    )

    # Fit scaler on this fold's training data only
    fold_scaler = StandardScaler()

    fold_train_scaled = fold_scaler.fit_transform(
        fold_train_processed
    )

    fold_valid_scaled = fold_scaler.transform(
        fold_valid_processed
    )

    # Fresh CatBoost model for this fold
    fold_model = CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=0
    )

    fold_model.fit(
        fold_train_scaled,
        y_fold_train
    )

    fold_pred = np.asarray(
        fold_model.predict(fold_valid_scaled)
    ).ravel()

    fold_prob = fold_model.predict_proba(
        fold_valid_scaled
    )[:, 1]

    cv_records.append({
        "Fold": fold_number,
        "Accuracy": accuracy_score(
            y_fold_valid, fold_pred
        ),
        "Precision": precision_score(
            y_fold_valid, fold_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_fold_valid, fold_pred, zero_division=0
        ),
        "F1 Score": f1_score(
            y_fold_valid, fold_pred, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_fold_valid, fold_prob
        )
    })

cv_fold_results = pd.DataFrame(cv_records)

cv_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],
    "Mean": [
        cv_fold_results["Accuracy"].mean(),
        cv_fold_results["Precision"].mean(),
        cv_fold_results["Recall"].mean(),
        cv_fold_results["F1 Score"].mean(),
        cv_fold_results["ROC-AUC"].mean()
    ],
    "Std": [
        cv_fold_results["Accuracy"].std(ddof=1),
        cv_fold_results["Precision"].std(ddof=1),
        cv_fold_results["Recall"].std(ddof=1),
        cv_fold_results["F1 Score"].std(ddof=1),
        cv_fold_results["ROC-AUC"].std(ddof=1)
    ]
})

print("Fold-level results:")
display(cv_fold_results)

print("Cross-validation summary:")
display(cv_summary)


In [ ]:
print("Leakage-safe CatBoost 5-Fold Cross-Validation")
for _, row in cv_summary.iterrows():
    print(
        f'{row["Metric"]}: '
        f'{row["Mean"]:.6f} ± {row["Std"]:.6f}'
    )


# Save Final Model and Preprocessing Artifacts

The files are saved inside a `models/` folder so the Streamlit application can load the exact same preprocessing objects and feature order used during training.

**Important:** The saved feature list does not contain `RiskScore`.


In [ ]:
import os
import joblib

os.makedirs("models", exist_ok=True)

joblib.dump(
    catboost_model,
    "models/final_catboost_model.pkl"
)

joblib.dump(
    scaler,
    "models/final_scaler.pkl"
)

joblib.dump(
    ohe,
    "models/final_ohe.pkl"
)

joblib.dump(
    num_imp,
    "models/final_num_imputer.pkl"
)

joblib.dump(
    cat_imp,
    "models/final_cat_imputer.pkl"
)

joblib.dump(
    feature_columns,
    "models/final_feature_columns.pkl"
)

joblib.dump(
    X_train_scaled,
    "models/final_X_train_scaled.pkl"
)

joblib.dump(
    model_numerical_cols,
    "models/final_numerical_columns.pkl"
)

joblib.dump(
    model_categorical_cols,
    "models/final_categorical_columns.pkl"
)

print("All final model artifacts saved in ./models/")
print("Number of final model features:", len(feature_columns))
print("RiskScore included:", "RiskScore" in feature_columns)


In [ ]:
# Verify saved artifacts
required_files = [
    "final_catboost_model.pkl",
    "final_scaler.pkl",
    "final_ohe.pkl",
    "final_num_imputer.pkl",
    "final_cat_imputer.pkl",
    "final_feature_columns.pkl",
    "final_X_train_scaled.pkl",
    "final_numerical_columns.pkl",
    "final_categorical_columns.pkl"
]

verification = {
    filename: os.path.exists(os.path.join("models", filename))
    for filename in required_files
}

pd.DataFrame(
    list(verification.items()),
    columns=["File", "Exists"]
)


# Explainable AI (SHAP + LIME)

SHAP and LIME are applied to the **corrected CatBoost model without RiskScore**.

The same transformed feature representation used by the model is used for the explanations.


In [ ]:
import shap

shap_explainer = shap.TreeExplainer(catboost_model)
shap_raw = shap_explainer.shap_values(X_test_scaled)

# Handle SHAP output consistently across SHAP versions.
if isinstance(shap_raw, list):
    shap_values = np.asarray(shap_raw[-1])
elif np.asarray(shap_raw).ndim == 3:
    shap_values = np.asarray(shap_raw)[:, :, -1]
else:
    shap_values = np.asarray(shap_raw)

print("SHAP matrix shape:", shap_values.shape)
print("Test matrix shape:", X_test_scaled.shape)


In [ ]:
# Global SHAP summary plot
plt.figure()
shap.summary_plot(
    shap_values,
    X_test_scaled,
    feature_names=feature_columns,
    show=False
)
plt.tight_layout()
plt.savefig(
    "shap_summary_plot.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()


In [ ]:
# Select representative correctly classified approved and rejected applicants.
# This makes the XAI examples consistent with their displayed outcome.

test_predictions = np.asarray(
    catboost_model.predict(X_test_scaled)
).ravel()

test_approval_prob = catboost_model.predict_proba(
    X_test_scaled
)[:, 1]

y_test_array = y_test.to_numpy()

approved_correct = np.where(
    (y_test_array == 1) &
    (test_predictions == 1)
)[0]

rejected_correct = np.where(
    (y_test_array == 0) &
    (test_predictions == 0)
)[0]

# Prefer correctly classified examples. If a class has no correct example,
# fall back to the highest-confidence sample from that true class.
if len(approved_correct) > 0:
    approved_index = approved_correct[
        np.argmax(test_approval_prob[approved_correct])
    ]
else:
    approved_candidates = np.where(y_test_array == 1)[0]
    approved_index = approved_candidates[
        np.argmax(test_approval_prob[approved_candidates])
    ]

if len(rejected_correct) > 0:
    rejected_index = rejected_correct[
        np.argmin(test_approval_prob[rejected_correct])
    ]
else:
    rejected_candidates = np.where(y_test_array == 0)[0]
    rejected_index = rejected_candidates[
        np.argmin(test_approval_prob[rejected_candidates])
    ]

approved_sample = X_test_scaled[approved_index]
rejected_sample = X_test_scaled[rejected_index]

print("Approved example:")
print("  Actual class:", y_test_array[approved_index])
print("  Predicted class:", test_predictions[approved_index])
print("  Approval probability:", test_approval_prob[approved_index])

print("Rejected example:")
print("  Actual class:", y_test_array[rejected_index])
print("  Predicted class:", test_predictions[rejected_index])
print("  Approval probability:", test_approval_prob[rejected_index])


In [ ]:
# Select representative high-confidence approved and rejected applicants
test_approval_prob = catboost_model.predict_proba(
    X_test_scaled
)[:, 1]

approved_candidates = np.where(y_test.to_numpy() == 1)[0]
rejected_candidates = np.where(y_test.to_numpy() == 0)[0]

if len(approved_candidates) == 0 or len(rejected_candidates) == 0:
    raise ValueError("Both approved and rejected samples are required for XAI examples.")

approved_index = approved_candidates[
    np.argmax(test_approval_prob[approved_candidates])
]

rejected_index = rejected_candidates[
    np.argmin(test_approval_prob[rejected_candidates])
]

approved_sample = X_test_scaled[approved_index]
rejected_sample = X_test_scaled[rejected_index]

print("Approved sample position:", approved_index)
print("Rejected sample position:", rejected_index)
print("Approved probability:", test_approval_prob[approved_index])
print("Rejected probability:", test_approval_prob[rejected_index])


In [ ]:
# SHAP local explanations
approved_base_value = shap_explainer.expected_value
rejected_base_value = shap_explainer.expected_value

if isinstance(approved_base_value, (list, np.ndarray)):
    approved_base_value = np.asarray(approved_base_value).reshape(-1)[-1]
    rejected_base_value = np.asarray(rejected_base_value).reshape(-1)[-1]

approved_shap_exp = shap.Explanation(
    values=shap_values[approved_index],
    base_values=approved_base_value,
    data=approved_sample,
    feature_names=feature_columns
)

rejected_shap_exp = shap.Explanation(
    values=shap_values[rejected_index],
    base_values=rejected_base_value,
    data=rejected_sample,
    feature_names=feature_columns
)

plt.figure(figsize=(9, 6))
shap.plots.waterfall(
    approved_shap_exp,
    max_display=10,
    show=False
)
plt.tight_layout()
plt.savefig(
    "shap_approved_waterfall.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()

plt.figure(figsize=(9, 6))
shap.plots.waterfall(
    rejected_shap_exp,
    max_display=10,
    show=False
)
plt.tight_layout()
plt.savefig(
    "shap_rejected_waterfall.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()


In [ ]:
# LIME
try:
    from lime.lime_tabular import LimeTabularExplainer
except ImportError:
    import sys
    import subprocess

    print("LIME is not installed. Attempting installation...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "lime", "-q"]
    )

    from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    X_train_scaled,
    feature_names=feature_columns,
    class_names=["Rejected", "Approved"],
    mode="classification",
    random_state=42
)

approved_lime_exp = lime_explainer.explain_instance(
    approved_sample,
    catboost_model.predict_proba,
    num_features=10,
    labels=(1,)
)

rejected_lime_exp = lime_explainer.explain_instance(
    rejected_sample,
    catboost_model.predict_proba,
    num_features=10,
    labels=(1,)
)

print("LIME explanations generated successfully.")


In [ ]:
# Save LIME plots correctly
approved_lime_fig = approved_lime_exp.as_pyplot_figure(label=1)
approved_lime_fig.tight_layout()
approved_lime_fig.savefig(
    "lime_approved_explanation.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close(approved_lime_fig)

rejected_lime_fig = rejected_lime_exp.as_pyplot_figure(label=1)
rejected_lime_fig.tight_layout()
rejected_lime_fig.savefig(
    "lime_rejected_explanation.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close(rejected_lime_fig)


# SHAP vs LIME Comparison

The comparison is performed on common feature names.  
Only the features explicitly identified by both explanation methods are compared.


In [ ]:
def compare_shap_lime(index, lime_exp):
    shap_df = pd.DataFrame({
        "Feature": feature_columns,
        "SHAP_Value": shap_values[index]
    })

    shap_df["Absolute_SHAP"] = shap_df["SHAP_Value"].abs()

    shap_df = (
        shap_df
        .sort_values("Absolute_SHAP", ascending=False)
        .head(10)
    )

    lime_data = []

    for condition, contribution in lime_exp.as_list(label=1):
        matching_features = [
            column for column in feature_columns
            if column in condition
        ]

        if matching_features:
            # Use the longest matching feature name to avoid
            # accidental partial matches.
            feature = max(
                matching_features,
                key=len
            )
            lime_data.append(
                [feature, contribution]
            )

    lime_df = pd.DataFrame(
        lime_data,
        columns=["Feature", "LIME_Contribution"]
    )

    comparison = pd.merge(
        shap_df[["Feature", "SHAP_Value"]],
        lime_df,
        on="Feature",
        how="inner"
    )

    if not comparison.empty:
        comparison["Direction_Agreement"] = (
            np.sign(comparison["SHAP_Value"]) ==
            np.sign(comparison["LIME_Contribution"])
        )
    else:
        comparison["Direction_Agreement"] = pd.Series(
            dtype=bool
        )

    return comparison

approved_comparison = compare_shap_lime(
    approved_index,
    approved_lime_exp
)

rejected_comparison = compare_shap_lime(
    rejected_index,
    rejected_lime_exp
)

print("APPROVED APPLICANT")
display(approved_comparison)

print("\nREJECTED APPLICANT")
display(rejected_comparison)


In [ ]:
# Directional agreement summary
def agreement_percentage(comparison):
    if len(comparison) == 0:
        return np.nan
    return comparison["Direction_Agreement"].mean() * 100

approved_agreement = agreement_percentage(approved_comparison)
rejected_agreement = agreement_percentage(rejected_comparison)

agreement_summary = pd.DataFrame({
    "Applicant": ["Approved", "Rejected"],
    "Common Features": [
        len(approved_comparison),
        len(rejected_comparison)
    ],
    "Directional Agreement (%)": [
        approved_agreement,
        rejected_agreement
    ]
})

agreement_summary


In [ ]:
# Save comparison tables
approved_comparison.to_csv(
    "SHAP_LIME_Approved_Comparison.csv",
    index=False
)

rejected_comparison.to_csv(
    "SHAP_LIME_Rejected_Comparison.csv",
    index=False
)

agreement_summary.to_csv(
    "SHAP_LIME_Agreement_Summary.csv",
    index=False
)

print("SHAP/LIME comparison files saved.")


# Final Prediction and Risk Assessment

Risk is calculated **after** the model prediction.

- Approval probability = probability of class 1
- Rejection probability = probability of class 0
- High Risk: rejection probability ≥ 0.50
- Medium Risk: 0.20 to < 0.50
- Low Risk: < 0.20

These thresholds are decision-support thresholds used by the application and are not additional model features.


In [ ]:
# Final model predictions for representative applicants

approved_prediction = int(
    catboost_model.predict(
        approved_sample.reshape(1, -1)
    )[0]
)

approved_probabilities = catboost_model.predict_proba(
    approved_sample.reshape(1, -1)
)[0]

rejected_prediction = int(
    catboost_model.predict(
        rejected_sample.reshape(1, -1)
    )[0]
)

rejected_probabilities = catboost_model.predict_proba(
    rejected_sample.reshape(1, -1)
)[0]

def risk_from_rejection_probability(rejection_probability):
    if rejection_probability >= 0.50:
        return "High Risk"
    elif rejection_probability >= 0.20:
        return "Medium Risk"
    return "Low Risk"

representative_results = pd.DataFrame({
    "Applicant": ["Approved Example", "Rejected Example"],
    "Predicted Class": [
        "Approved" if approved_prediction == 1 else "Rejected",
        "Approved" if rejected_prediction == 1 else "Rejected"
    ],
    "Approval Probability (%)": [
        approved_probabilities[1] * 100,
        rejected_probabilities[1] * 100
    ],
    "Rejection Probability (%)": [
        approved_probabilities[0] * 100,
        rejected_probabilities[0] * 100
    ],
    "Risk Level": [
        risk_from_rejection_probability(approved_probabilities[0]),
        risk_from_rejection_probability(rejected_probabilities[0])
    ]
})

representative_results


# Final Verification Checklist

Before using the outputs in the research paper, verify the following:

- `RiskScore` is not in `feature_columns`.
- The target `LoanApproved` is not in `feature_columns`.
- Imputers are fitted only on `X_train_raw`.
- OneHotEncoder is fitted only on training categorical data.
- StandardScaler is fitted only on training data.
- Test data is transformed, not fitted.
- Cross-validation refits preprocessing inside every fold.
- SHAP and LIME use the corrected CatBoost model.
- Saved artifacts are located in `models/`.


In [ ]:
# Automated final checks
assert TARGET not in feature_columns, "Target leakage: LoanApproved is in model features."
assert "RiskScore" not in feature_columns, "Leakage risk: RiskScore is in model features."
assert X_train_scaled.shape[1] == len(feature_columns)
assert X_test_scaled.shape[1] == len(feature_columns)
assert len(y_train) == X_train_scaled.shape[0]
assert len(y_test) == X_test_scaled.shape[0]

print("ALL FINAL CHECKS PASSED.")
print("Final feature count:", len(feature_columns))
print("RiskScore excluded:", "RiskScore" not in feature_columns)
print("Target excluded:", TARGET not in feature_columns)
print("Saved model path:", os.path.exists("models/final_catboost_model.pkl"))
